# 🚀 PMMPL Production AI - RAG System Experimentation

## Project Overview
This notebook contains the **core logic and experimentation** for building a RAG (Retrieval-Augmented Generation) system using **LangGraph** for production management with Google Sheets data.

### Key Features:
- 📊 **Multi-Sheet Google Sheets Integration**
- 🤖 **LangGraph Agent** with Plan-Execute-Answer architecture
- 🔍 **Semantic Sheet Selection** using embeddings
- 💾 **Intelligent Caching** with similarity matching
- 🎯 **7 Operation Types**: Retrieve, Aggregate, Compare, Feasibility, Rank, Trend, Predict

### Use Cases:
- Query production data using natural language
- Analyze orders, stock, payments across multiple sheets
- Check feasibility of orders vs stock
- Rank top products/customers
- Aggregate revenue, quantities, etc.

---

**Author**: PMMPL Production AI Team  
**Version**: 2.0  
**Date**: January 2026

## 1️⃣ Setup and Imports

Install and import all required libraries for the RAG system.

In [ ]:
# Core Libraries
import os
import json
import pandas as pd
import numpy as np
from typing import TypedDict, Annotated, Optional, Dict, Any, List
from datetime import datetime, timedelta

# LangChain & LangGraph
from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END

# Google Sheets
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# Embeddings & Similarity
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Utilities
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

## 2️⃣ Configuration

Set up API keys and environment variables.

In [ ]:
# Load environment variables
load_dotenv()

# API Configuration
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "your-groq-api-key-here")
GOOGLE_SHEETS_URL = os.getenv("GOOGLE_SHEETS_URL", "your-google-sheets-url")

# LLM Configuration
LLM_MODEL = "llama-3.3-70b-versatile"
LLM_TEMPERATURE = 0.0

# Embedding Model
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# Cache Settings
CACHE_SIMILARITY_THRESHOLD = 0.98

print(f"✅ Configuration loaded")
print(f"   LLM: {LLM_MODEL}")
print(f"   Embedding Model: {EMBEDDING_MODEL}")
print(f"   Cache Threshold: {CACHE_SIMILARITY_THRESHOLD}")

## 3️⃣ Google Sheets Data Loading

Connect to Google Sheets and load data as pandas DataFrames.

In [ ]:
def load_google_sheets_data(credentials_file: str, sheet_url: str) -> Dict[str, pd.DataFrame]:
    """
    Load all worksheets from a Google Sheet into pandas DataFrames.
    
    Args:
        credentials_file: Path to Google Service Account JSON
        sheet_url: Google Sheets URL
    
    Returns:
        Dictionary with sheet names as keys and DataFrames as values
    """
    # Authenticate
    scope = ['https://spreadsheets.google.com/feeds', 
             'https://www.googleapis.com/auth/drive']
    creds = ServiceAccountCredentials.from_json_keyfile_name(credentials_file, scope)
    client = gspread.authorize(creds)
    
    # Open spreadsheet
    spreadsheet = client.open_by_url(sheet_url)
    
    # Load all sheets
    dataframes = {}
    total_rows = 0
    
    for worksheet in spreadsheet.worksheets():
        sheet_name = worksheet.title
        data = worksheet.get_all_records()
        df = pd.DataFrame(data)
        
        if not df.empty:
            dataframes[sheet_name] = df
            total_rows += len(df)
            print(f"✓ Loaded '{sheet_name}': {len(df)} rows, {len(df.columns)} columns")
    
    print(f"\n✅ Total: {len(dataframes)} sheets, {total_rows} rows")
    return dataframes

# Example usage (update with your credentials path)
# dataframes = load_google_sheets_data('credentials.json', GOOGLE_SHEETS_URL)

# For demonstration, create sample data
dataframes = {
    "Orders Pending": pd.DataFrame({
        "DO-Delivery Order No.": ["DO-2072", "DO-2075", "DO-2222"],
        "Party Names": ["Rungta Mines", "Rungta Mines", "Singhal Steel"],
        "Product Name": ["Dura Cast", "Pasheat - K", "Dura Special"],
        "Quantity": [65, 20, 30],
        "Rate Of Material": [60000, 26500, 36500],
        "Pending Qty": [57.5, 20, 2],
        "Status": ["Pending", "Pending", "Pending"]
    }),
    "FG Stock": pd.DataFrame({
        "Product Name": ["Dura Cast", "Pasheat - K", "Dura Special"],
        "Current Level": [65, 6.37, 12.317],
        "Unit": ["MT", "MT", "MT"]
    })
}

print("\n📊 Sample data loaded for demonstration")

## 4️⃣ Embeddings for Sheet Selection

Use sentence transformers to generate embeddings and select relevant sheets based on query similarity.

In [ ]:
class SheetSelector:
    """Select relevant sheets using semantic similarity"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.sheet_embeddings = {}
        self.sheet_descriptions = {}
    
    def generate_embeddings(self, dataframes: Dict[str, pd.DataFrame]):
        """Generate embeddings for each sheet"""
        for name, df in dataframes.items():
            # Create description from sheet name and columns
            description = f"{name}: {', '.join(df.columns.tolist())}"
            self.sheet_descriptions[name] = description
            self.sheet_embeddings[name] = self.model.encode(description)
        
        print(f"✅ Generated embeddings for {len(dataframes)} sheets")
    
    def select_top_sheets(self, query: str, top_n: int = 3) -> List[str]:
        """Select top N most relevant sheets for the query"""
        query_embedding = self.model.encode(query)
        
        similarities = {}
        for name, embedding in self.sheet_embeddings.items():
            sim = cosine_similarity(
                query_embedding.reshape(1, -1),
                embedding.reshape(1, -1)
            )[0][0]
            similarities[name] = sim
        
        # Sort and return top N
        top_sheets = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:top_n]
        
        print(f"\n🎯 Selected top {top_n} sheets for: '{query}'")
        for sheet, score in top_sheets:
            print(f"   • {sheet}: {score:.2%}")
        
        return [sheet for sheet, _ in top_sheets]

# Initialize and test
selector = SheetSelector(EMBEDDING_MODEL)
selector.generate_embeddings(dataframes)

# Test queries
test_queries = [
    "Show me pending orders",
    "Check current stock levels",
    "Can we fulfill orders with stock?"
]

for query in test_queries:
    selected = selector.select_top_sheets(query, top_n=2)
    print()

## 5️⃣ LangGraph State Definition

Define the state structure for our agent using TypedDict.

In [ ]:
class AgentState(TypedDict):
    """State for the LangGraph agent"""
    question: str                    # User's natural language query
    plan: Optional[Dict[str, Any]]   # Execution plan from LLM
    result: Optional[Any]            # Query execution result (DataFrame)
    answer: str                      # Natural language answer
    confidence: float                # Confidence score (0.0 to 1.0)
    error: Optional[str]             # Error message if any
    retry_count: int                 # Number of retries
    selected_sheets: List[str]       # Sheets selected for this query

print("✅ Agent state structure defined")
print("\nState Fields:")
for field, field_type in AgentState.__annotations__.items():
    print(f"   • {field}: {field_type}")

## 6️⃣ Operation Types Implementation

Implement the 7 core operation types: RETRIEVE, AGGREGATE, COMPARE, FEASIBILITY, RANK, TREND, PREDICT

In [ ]:
class DataOperations:
    """Execute different types of data operations"""
    
    @staticmethod
    def retrieve(df: pd.DataFrame, filters: Optional[str] = None) -> pd.DataFrame:
        """RETRIEVE: Filter and return data"""
        if filters:
            try:
                result = df.query(filters)
                print(f"  ✓ Retrieved {len(result)} rows with filter: {filters}")
                return result
            except Exception as e:
                print(f"  ⚠️ Filter error: {e}")
                return df
        return df
    
    @staticmethod
    def aggregate(df: pd.DataFrame, agg_func: str, agg_column: str) -> Dict[str, Any]:
        """AGGREGATE: SUM, COUNT, AVG, MAX, MIN"""
        agg_func = agg_func.upper()
        
        if agg_func == "COUNT":
            result = len(df)
        elif agg_func == "SUM":
            result = df[agg_column].sum()
        elif agg_func == "AVG":
            result = df[agg_column].mean()
        elif agg_func == "MAX":
            result = df[agg_column].max()
        elif agg_func == "MIN":
            result = df[agg_column].min()
        else:
            result = None
        
        print(f"  ✓ {agg_func}({agg_column}) = {result}")
        return {"result": result, "function": agg_func, "column": agg_column}
    
    @staticmethod
    def compare(df1: pd.DataFrame, df2: pd.DataFrame, join_key: str) -> pd.DataFrame:
        """COMPARE: Join two sheets"""
        result = df1.merge(df2, on=join_key, how="inner")
        print(f"  ✓ Joined on '{join_key}': {len(result)} rows")
        return result
    
    @staticmethod
    def feasibility(df1: pd.DataFrame, df2: pd.DataFrame, join_key: str) -> pd.DataFrame:
        """FEASIBILITY: Check if conditions can be met"""
        result = df1.merge(df2, on=join_key, how="inner")
        
        # Example: Check if Current Level >= Pending Qty
        if 'Current Level' in result.columns and 'Pending Qty' in result.columns:
            result['Can_Fulfill'] = result['Current Level'] >= result['Pending Qty']
            result['Gap'] = result['Current Level'] - result['Pending Qty']
        
        print(f"  ✓ Feasibility check complete: {len(result)} rows")
        return result
    
    @staticmethod
    def rank(df: pd.DataFrame, rank_by: str, limit: int = 10, ascending: bool = False) -> pd.DataFrame:
        """RANK: Top N or Bottom N"""
        if rank_by == "COUNT":
            # Count-based ranking (e.g., party with most orders)
            group_col = df.columns[0]  # First column as default
            result = df.groupby(group_col).size().reset_index(name='Count')
            result = result.sort_values('Count', ascending=ascending).head(limit)
        else:
            result = df.sort_values(rank_by, ascending=ascending).head(limit)
        
        print(f"  ✓ Ranked by '{rank_by}', top {limit}")
        return result

# Test operations
ops = DataOperations()

# Test 1: RETRIEVE
print("\n1️⃣ RETRIEVE - Filter pending orders:")
result1 = ops.retrieve(dataframes["Orders Pending"], "`Pending Qty` > 10")
print(result1[['Product Name', 'Pending Qty']])

# Test 2: AGGREGATE
print("\n\n2️⃣ AGGREGATE - Total pending quantity:")
result2 = ops.aggregate(dataframes["Orders Pending"], "SUM", "Pending Qty")

# Test 3: RANK
print("\n\n3️⃣ RANK - Top 2 products by pending quantity:")
result3 = ops.rank(dataframes["Orders Pending"], "Pending Qty", limit=2)
print(result3[['Product Name', 'Pending Qty']])

## 7️⃣ LLM Integration - Planner

Create the planner that converts natural language queries into structured execution plans.

In [ ]:
# Initialize LLM
llm = ChatGroq(model=LLM_MODEL, temperature=LLM_TEMPERATURE, api_key=GROQ_API_KEY)

# Planner prompt template
PLANNER_PROMPT = """You are a query planning expert for production management data.

Available sheets:
{sheets_info}

Convert the user's question into a JSON execution plan.

Operation Types:
1. retrieve: Filter and return data
2. aggregate: SUM, COUNT, AVG, MAX, MIN
3. compare: Join multiple sheets
4. feasibility: Check if conditions can be met
5. rank: Top N or Bottom N

Return ONLY JSON:
{{
  "sheets": ["Sheet Name"],
  "operation": "retrieve|aggregate|compare|feasibility|rank",
  "filters": {{"SheetName": "filter_expression"}},
  "rank_by": "column_name",
  "rank_order": "desc|asc",
  "limit": 10
}}

Question: {question}
"""

def create_plan(question: str, sheets_info: str) -> Dict[str, Any]:
    """Generate execution plan using LLM"""
    prompt = ChatPromptTemplate.from_template(PLANNER_PROMPT)
    messages = prompt.format_messages(question=question, sheets_info=sheets_info)
    
    response = llm.invoke(messages)
    
    # Parse JSON from response
    content = response.content
    if "```json" in content:
        content = content.split("```json")[1].split("```")[0]
    elif "```" in content:
        content = content.split("```")[1].split("```")[0]
    
    plan = json.loads(content.strip())
    
    print(f"📋 Plan generated:")
    print(f"   Operation: {plan.get('operation')}")
    print(f"   Sheets: {plan.get('sheets')}")
    
    return plan

# Test planner
sheets_info = "\n".join([f"- {name}: {list(df.columns)}" for name, df in dataframes.items()])

test_questions = [
    "Show me all pending orders",
    "Top 5 products by pending quantity",
    "Can we fulfill orders with current stock?"
]

for question in test_questions:
    print(f"\n\n❓ Question: {question}")
    plan = create_plan(question, sheets_info)
    print(f"   {json.dumps(plan, indent=2)}")

## 8️⃣ LangGraph Agent - Node Functions

Define the three core nodes: PLAN → EXECUTE → ANSWER

In [ ]:
class RAGAgent:
    """Complete RAG Agent with LangGraph"""
    
    def __init__(self, dataframes: Dict[str, pd.DataFrame], llm, selector: SheetSelector):
        self.dataframes = dataframes
        self.llm = llm
        self.selector = selector
        self.ops = DataOperations()
    
    def plan_node(self, state: AgentState) -> dict:
        """Node 1: Generate execution plan"""
        print("\n🎯 Node: PLAN")
        
        # Select relevant sheets
        selected_sheets = self.selector.select_top_sheets(state["question"], top_n=2)
        
        # Generate plan
        sheets_info = "\n".join([
            f"- {name}: {list(self.dataframes[name].columns)}" 
            for name in selected_sheets
        ])
        
        plan = create_plan(state["question"], sheets_info)
        
        return {
            "plan": plan,
            "selected_sheets": selected_sheets
        }
    
    def execute_node(self, state: AgentState) -> dict:
        """Node 2: Execute the plan"""
        print("\n🎯 Node: EXECUTE")
        
        plan = state["plan"]
        operation = plan.get("operation")
        
        # Get DataFrames for selected sheets
        dfs = {name: self.dataframes[name] for name in plan["sheets"]}
        
        # Execute based on operation type
        try:
            if operation == "retrieve":
                df = list(dfs.values())[0]
                filters = plan.get("filters", {}).get(plan["sheets"][0])
                result = self.ops.retrieve(df, filters)
            
            elif operation == "aggregate":
                df = list(dfs.values())[0]
                result = self.ops.aggregate(
                    df,
                    plan.get("agg_function", "COUNT"),
                    plan.get("agg_column", "")
                )
            
            elif operation == "rank":
                df = list(dfs.values())[0]
                result = self.ops.rank(
                    df,
                    plan.get("rank_by", ""),
                    plan.get("limit", 10),
                    plan.get("rank_order", "desc") == "asc"
                )
            
            elif operation == "feasibility":
                result = self.ops.feasibility(
                    list(dfs.values())[0],
                    list(dfs.values())[1],
                    plan.get("join_key", "Product Name")
                )
            
            else:
                result = list(dfs.values())[0]
            
            return {"result": result, "error": None}
        
        except Exception as e:
            print(f"  ⚠️ Execution error: {e}")
            return {"result": None, "error": str(e)}
    
    def answer_node(self, state: AgentState) -> dict:
        """Node 3: Generate natural language answer"""
        print("\n🎯 Node: ANSWER")
        
        result = state["result"]
        
        if isinstance(result, pd.DataFrame):
            summary = f"Found {len(result)} rows"
            if len(result) > 0:
                summary += f"\\nColumns: {', '.join(result.columns.tolist())}"
        elif isinstance(result, dict):
            summary = f"{result['function']} = {result['result']}"
        else:
            summary = str(result)
        
        # Generate answer using LLM
        answer_prompt = f"""Based on the query results, provide a natural language answer.

Question: {state['question']}
Results: {summary}

Provide a clear, concise answer:"""
        
        response = self.llm.invoke(answer_prompt)
        answer = response.content
        
        print(f"  ✓ Answer generated ({len(answer)} chars)")
        
        return {
            "answer": answer,
            "confidence": 0.9 if state["error"] is None else 0.3
        }

# Initialize agent
agent = RAGAgent(dataframes, llm, selector)

print("✅ RAG Agent initialized with 3 nodes: PLAN → EXECUTE → ANSWER")

## 9️⃣ Build LangGraph Workflow

Construct the graph with nodes and edges.

In [ ]:
# Build the graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("plan", agent.plan_node)
workflow.add_node("execute", agent.execute_node)
workflow.add_node("answer", agent.answer_node)

# Set entry point
workflow.set_entry_point("plan")

# Add edges: PLAN → EXECUTE → ANSWER → END
workflow.add_edge("plan", "execute")
workflow.add_edge("execute", "answer")
workflow.add_edge("answer", END)

# Compile the graph
compiled_agent = workflow.compile()

print("✅ LangGraph workflow built successfully!")
print("\nWorkflow: PLAN → EXECUTE → ANSWER → END")

## 🔟 Semantic Caching System

Implement query caching with embeddings for faster responses.

In [ ]:
class QueryCache:
    """Semantic caching for query results"""
    
    def __init__(self, model: SentenceTransformer, threshold: float = 0.98):
        self.model = model
        self.threshold = threshold
        self.cache = {}  # {query: {"embedding": np.array, "result": Any, "timestamp": datetime}}
    
    def check_cache(self, query: str) -> Optional[Any]:
        """Check if similar query exists in cache"""
        if not self.cache:
            return None
        
        query_embedding = self.model.encode(query)
        
        best_match = None
        best_similarity = 0.0
        
        for cached_query, cached_data in self.cache.items():
            similarity = cosine_similarity(
                query_embedding.reshape(1, -1),
                cached_data["embedding"].reshape(1, -1)
            )[0][0]
            
            if similarity > best_similarity and similarity >= self.threshold:
                best_similarity = similarity
                best_match = cached_query
        
        if best_match:
            print(f"💾 Cache HIT! Similarity: {best_similarity:.2%}")
            return self.cache[best_match]["result"]
        
        print(f"❌ Cache MISS (best: {best_similarity:.2%})")
        return None
    
    def save_to_cache(self, query: str, result: Any):
        """Save query result to cache"""
        embedding = self.model.encode(query)
        self.cache[query] = {
            "embedding": embedding,
            "result": result,
            "timestamp": datetime.now()
        }
        print(f"💾 Saved to cache: '{query[:50]}...'")
    
    def clear_cache(self):
        """Clear all cached queries"""
        self.cache = {}
        print("🗑️ Cache cleared")

# Initialize cache
cache = QueryCache(selector.model, threshold=CACHE_SIMILARITY_THRESHOLD)

# Test caching
print("\n📝 Test 1: First query (cache miss expected)")
test_query1 = "Show me pending orders"
cached_result = cache.check_cache(test_query1)

# Simulate saving result
cache.save_to_cache(test_query1, {"data": "sample_result"})

print("\n\n📝 Test 2: Similar query (cache hit expected)")
test_query2 = "Show pending orders"  # Very similar
cached_result = cache.check_cache(test_query2)

print("\n\n📝 Test 3: Different query (cache miss expected)")
test_query3 = "What is the total revenue?"  # Different
cached_result = cache.check_cache(test_query3)

print(f"\n✅ Cache system ready (threshold: {CACHE_SIMILARITY_THRESHOLD})")

## 1️⃣1️⃣ End-to-End Query Execution

Test the complete RAG system with real queries.

In [ ]:
def query_with_cache(question: str) -> Dict[str, Any]:
    """Execute query with caching"""
    
    # Check cache first
    cached_result = cache.check_cache(question)
    if cached_result:
        return cached_result
    
    # Execute through agent
    initial_state = {
        "question": question,
        "plan": None,
        "result": None,
        "answer": "",
        "confidence": 0.0,
        "error": None,
        "retry_count": 0,
        "selected_sheets": []
    }
    
    result = compiled_agent.invoke(initial_state)
    
    # Save to cache
    cache.save_to_cache(question, result)
    
    return result

# Test queries
test_queries = [
    "Show me all pending orders",
    "Top 3 products by pending quantity",
    "Show me pending orders",  # Duplicate to test cache
]

for i, question in enumerate(test_queries, 1):
    print(f"\n\n{'='*80}")
    print(f"Query {i}: {question}")
    print('='*80)
    
    result = query_with_cache(question)
    
    print(f"\n📊 RESULT:")
    print(f"   Confidence: {result['confidence']}")
    print(f"   Answer: {result['answer'][:200]}...")
    
    if isinstance(result.get('result'), pd.DataFrame):
        print(f"   Rows: {len(result['result'])}")
        print(f"\n   Data Preview:")
        print(result['result'].head(3))

## 1️⃣2️⃣ Key Learnings & Reusable Patterns

Summary of core concepts for reusing this logic in other projects.

### 🎯 Reusable Patterns

#### 1. **LangGraph State Machine Pattern**
```python
# Define state
class MyState(TypedDict):
    input: str
    intermediate: Any
    output: str

# Create workflow
workflow = StateGraph(MyState)
workflow.add_node("node1", func1)
workflow.add_node("node2", func2)
workflow.set_entry_point("node1")
workflow.add_edge("node1", "node2")
workflow.add_edge("node2", END)
agent = workflow.compile()
```

#### 2. **Embedding-Based Selection Pattern**
```python
# Encode candidates
embeddings = {name: model.encode(text) for name, text in candidates.items()}

# Find best match
query_emb = model.encode(query)
similarities = {name: cosine_similarity(query_emb, emb) for name, emb in embeddings.items()}
best = max(similarities, key=similarities.get)
```

#### 3. **Semantic Cache Pattern**
```python
def check_cache(query, threshold=0.98):
    query_emb = model.encode(query)
    for cached_query, data in cache.items():
        sim = cosine_similarity(query_emb, data["embedding"])
        if sim >= threshold:
            return data["result"]
    return None
```

#### 4. **LLM JSON Planning Pattern**
```python
prompt = f"""Convert query to JSON:
Question: {question}
Return JSON: {{"operation": "...", "params": {{}}}}"""

response = llm.invoke(prompt)
plan = json.loads(response.content)
```

---

### 📋 Checklist for New Projects

- [ ] Define your data sources (SQL, APIs, files)
- [ ] Create operation types for your domain
- [ ] Design state structure for your workflow
- [ ] Implement 3 core nodes: PLAN, EXECUTE, ANSWER
- [ ] Add caching for repeated queries
- [ ] Test with domain-specific queries

## 1️⃣3️⃣ Requirements & Installation

Copy this to your `requirements.txt` for new projects.

In [ ]:
# Generate requirements.txt content
requirements = """
# Core LangChain & LangGraph
langchain>=0.1.0
langchain-groq>=0.0.1
langgraph>=0.2.0

# Google Sheets
gspread>=5.12.0
oauth2client>=4.1.3

# Embeddings
sentence-transformers>=2.2.2
scikit-learn>=1.3.0

# Data Processing
pandas>=2.0.0
numpy>=1.24.0

# API & Server (for FastAPI backend)
fastapi>=0.109.0
uvicorn>=0.25.0
python-dotenv>=1.0.0

# Database (for caching)
sqlalchemy>=2.0.0
"""

print("📋 requirements.txt:")
print(requirements)

# Installation command
print("\n\n⚙️ Installation Command:")
print("pip install langchain langchain-groq langgraph gspread oauth2client sentence-transformers scikit-learn pandas numpy fastapi uvicorn python-dotenv sqlalchemy")

## 1️⃣4️⃣ Architecture Summary

```
┌─────────────────────────────────────────────────────────────────┐
│                    PMMPL RAG System Architecture                 │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│  User Query: "Show me pending orders"                           │
│       ↓                                                          │
│  ┌─────────────┐                                                │
│  │   CACHE     │ ← Check semantic similarity (threshold: 98%)   │
│  └──────┬──────┘                                                │
│         │ Miss                                                   │
│         ↓                                                        │
│  ┌─────────────┐                                                │
│  │ EMBEDDINGS  │ ← Select top 3 relevant sheets                 │
│  └──────┬──────┘                                                │
│         ↓                                                        │
│  ╔═══════════════════════════════════════════════════════════╗  │
│  ║                   LANGGRAPH WORKFLOW                       ║  │
│  ║  ┌──────────┐   ┌───────────┐   ┌──────────┐             ║  │
│  ║  │   PLAN   │ → │  EXECUTE  │ → │  ANSWER  │             ║  │
│  ║  │  (LLM)   │   │  (Pandas) │   │  (LLM)   │             ║  │
│  ║  └──────────┘   └───────────┘   └──────────┘             ║  │
│  ║                                                           ║  │
│  ║  Operations: RETRIEVE | AGGREGATE | COMPARE |             ║  │
│  ║              FEASIBILITY | RANK | TREND | PREDICT        ║  │
│  ╚═══════════════════════════════════════════════════════════╝  │
│         ↓                                                        │
│  ┌─────────────┐                                                │
│  │   CACHE     │ ← Save result for future queries              │
│  └──────┬──────┘                                                │
│         ↓                                                        │
│  Response: Natural language + Data table                        │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

---

## 🏁 Conclusion

This notebook demonstrates the core components of a production-ready RAG system:

1. **Data Loading** - Multi-sheet Google Sheets integration
2. **Smart Selection** - Embeddings for relevant data selection
3. **Query Planning** - LLM converts natural language to structured plans
4. **Execution** - 7 operation types for comprehensive data analysis
5. **Caching** - Semantic similarity for performance optimization
6. **Answer Generation** - Natural language responses

Use these patterns as building blocks for your own RAG applications!